# 02 · Features & Class Imbalance

The engineered feature matrix and how we handle the fact that fires are **rare**.

> Run `uv run python scripts/02_features.py` first.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from wildfire.config import load_config
from wildfire.features.build import feature_columns
from wildfire.features.sampling import undersample_negatives, prior_correction

cfg = load_config()
df = pd.read_parquet(cfg.path_for('data_processed') / 'features.parquet')
df['date'] = pd.to_datetime(df['date'])
feats = feature_columns(df)
print(f'{len(df):,} rows · {len(feats)} model features')
print('features:', feats)

## The rare-event problem

Only a few percent of cell-weeks burn. A model that always predicts 'no fire' is
~97% accurate and useless — which is why we never use accuracy as a target metric.

In [ ]:
rate = df['fire'].mean()
print(f'positive rate: {rate*100:.2f}%  (1 fire per ~{1/rate:.0f} cell-weeks)')

# Undersampling for tractable training, then prior-correct probabilities back.
us = undersample_negatives(df, neg_per_pos=5.0, seed=cfg.seed)
print(f'after undersampling: {len(us):,} rows, positive rate {us["fire"].mean()*100:.1f}%')
demo_p = np.array([0.2, 0.5, 0.8])
print('example prior-correction (train rate 0.17 -> true rate %.3f):' % rate)
print('  model prob :', demo_p)
print('  calibrated :', np.round(prior_correction(demo_p, us['fire'].mean(), rate), 3))

## Feature correlations

A quick look at how the fire-domain features relate (and to the target).

In [ ]:
cols = [c for c in ['vpd','erc','bi','pdsi','tmax','rmin','wind','precip',
                    'days_since_rain','ndvi','fuel_load','lightning_density',
                    'human_proxy','fire'] if c in df.columns]
sample = df[cols].sample(min(50000, len(df)), random_state=0)
corr = sample.corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
fig.colorbar(im, shrink=0.7); ax.set_title('Feature correlation'); plt.tight_layout()

## Antecedent weather windows (dryness memory)

Fire risk depends not just on today's weather but on the recent trajectory. We
engineer rolling means/sums over the previous 2/4/8 steps.

In [ ]:
roll_cols = [c for c in df.columns if c.startswith('erc_roll') or c == 'erc']
cell = df['cell_id'].value_counts().index[0]
one = df[df['cell_id'] == cell].sort_values('date')
fig, ax = plt.subplots(figsize=(12, 4))
for c in roll_cols:
    ax.plot(one['date'], one[c], label=c, alpha=0.8)
ax.legend(); ax.set_title(f'ERC and antecedent windows · cell {cell}')
ax.set_ylabel('ERC'); plt.tight_layout()

**Next:** `03_modeling_and_validation.ipynb` — baselines vs the GBM under honest CV.